In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import zipfile
import os

# Διαδρομή προς το zip στο Drive σου
zip_path = '/content/drive/MyDrive/Pneumonia_detection/archive.zip'

# Προορισμός ξεζιπαρίσματος (το /content είναι ο τοπικός χώρος του Colab)
extract_path = '/content/chest_xray'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Το ξεζιπάρισμα ολοκληρώθηκε!")
print(os.listdir(extract_path)) # Θα πρέπει να δεις τους φακέλους train, test, val

Το ξεζιπάρισμα ολοκληρώθηκε!
['chest_xray']


In [1]:
import torch

# Ορίζουμε τη συσκευή (device). Αν υπάρχει GPU (cuda), τη χρησιμοποιούμε. Αλλιώς, CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Το notebook τρέχει σε: {device}")

Το notebook τρέχει σε: cuda


In [6]:
import os
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Ορίζουμε τα paths με βάση το δικό σου ξεζιπάρισμα
base_dir = '/content/chest_xray/chest_xray'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

In [7]:
# Μετασχηματισμοί για τις εικόνες (Data Augmentation για το train)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Για το validation και το test θέλουμε μόνο resize και normalization (όχι αλλοιώσεις)
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [8]:
# Φορτώνουμε τις εικόνες από τους φακέλους
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(root=val_dir, transform=val_test_transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=val_test_transform)

# Δημιουργούμε τους DataLoaders για να τροφοδοτούμε το μοντέλο σε batches των 32 εικόνων
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 163
Validation batches: 1
Test batches: 20


In [9]:
import torch.nn as nn
from torchvision import models

# 1. Κατεβάζουμε το προ-εκπαιδευμένο ResNet18
# Χρησιμοποιούμε τα standard βάρη (weights) του ImageNet
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# 2. "Παγώνουμε" (freeze) όλα τα επίπεδα του δικτύου
# Δεν θέλουμε να αλλάξουν οι βασικές γνώσεις που έχει ήδη το μοντέλο για σχήματα/γραμμές
for param in model.parameters():
    param.requires_grad = False

# 3. Αντικαθιστούμε το τελευταίο Fully Connected επίπεδο (fc)
# Το ResNet18 έχει model.fc.in_features εισόδους. Εμείς θέλουμε 2 εξόδους (Υγιής vs Πνευμονία)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)

# 4. Στέλνουμε το μοντέλο στην GPU (cuda) που ενεργοποιήσαμε πριν
model = model.to(device)

print("Το μοντέλο τροποποιήθηκε και μεταφέρθηκε στην GPU με επιτυχία!")
print(model.fc)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 109MB/s]


Το μοντέλο τροποποιήθηκε και μεταφέρθηκε στην GPU με επιτυχία!
Linear(in_features=512, out_features=2, bias=True)


In [11]:
import torch.optim as optim
import time

# 1. Ορίζουμε τον Κριτήριο Λάθους (Loss) και τον Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001) # Εκπαιδεύουμε μόνο το τελευταίο επίπεδο

num_epochs = 5
print("Έναρξη εκπαίδευσης...\n")

for epoch in range(num_epochs):
    start_time = time.time()

    # --- ΦΑΣΗ ΕΚΠΑΙΔΕΥΣΗΣ (TRAINING) ---
    model.train()
    running_loss = 0.0
    running_corrects = 0
    total_train_samples = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad() # Μηδενισμός των gradients

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        _, preds = torch.max(outputs, 1)

        loss.backward() # Backpropagation
        optimizer.step() # Ανανέωση βαρών

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        total_train_samples += inputs.size(0)

    epoch_loss = running_loss / total_train_samples
    epoch_acc = running_corrects.double() / total_train_samples

    # --- ΦΑΣΗ ΕΛΕΓΧΟΥ (VALIDATION / TEST LOADER) ---
    model.eval()
    val_loss = 0.0
    val_corrects = 0
    total_val_samples = 0

    with torch.no_grad(): # Κλείνουμε τα gradients για εξοικονόμηση μνήμης
        for inputs, labels in test_loader: # Χρησιμοποιούμε το test_loader για σωστό validation μεγέθους
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            total_val_samples += inputs.size(0)

    epoch_val_loss = val_loss / total_val_samples
    epoch_val_acc = val_corrects.double() / total_val_samples

    duration = time.time() - start_time
    print(f"Epoch {epoch+1}/{num_epochs} ({duration:.0f}s) -> "
          f"Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f} || "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")

print("\nΗ εκπαίδευση ολοκληρώθηκε με επιτυχία!")

Έναρξη εκπαίδευσης...

Epoch 1/5 (104s) -> Train Loss: 0.1393 | Train Acc: 0.9442 || Val Loss: 0.4939 | Val Acc: 0.8317


KeyboardInterrupt: 

In [12]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.eval() # Βάζουμε το μοντέλο σε evaluation mode

all_preds = []
all_labels = []

# Μαζεύουμε όλες τις προβλέψεις από το test_loader
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        # Μεταφέρουμε στην CPU για το scikit-learn
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# 1. Εκτύπωση του Classification Report (Precision, Recall, F1-Score)
target_names = ['NORMAL', 'PNEUMONIA']
print("--- MEDICAL CLASSIFICATION REPORT ---")
print(classification_report(all_labels, all_preds, target_names=target_names))

# 2. Εκτύπωση του Confusion Matrix (Πίνακας Σύγχυσης)
cm = confusion_matrix(all_labels, all_preds)
print("--- CONFUSION MATRIX ---")
print(f"True Negative (Υγιείς που βρέθηκαν Υγιείς): {cm[0][0]}")
print(f"False Positive (Υγιείς που βρέθηκαν με Πνευμονία): {cm[0][1]}")
print(f"False Negative (Ασθενείς που βρέθηκαν Υγιείς -> ΚΡΙΣΙΜΟ ΛΑΘΟΣ): {cm[1][0]}")
print(f"True Positive (Ασθενείς που βρέθηκαν με Πνευμονία): {cm[1][1]}")

--- MEDICAL CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

      NORMAL       0.93      0.73      0.82       234
   PNEUMONIA       0.86      0.97      0.91       390

    accuracy                           0.88       624
   macro avg       0.90      0.85      0.86       624
weighted avg       0.89      0.88      0.88       624

--- CONFUSION MATRIX ---
True Negative (Υγιείς που βρέθηκαν Υγιείς): 171
False Positive (Υγιείς που βρέθηκαν με Πνευμονία): 63
False Negative (Ασθενείς που βρέθηκαν Υγιείς -> ΚΡΙΣΙΜΟ ΛΑΘΟΣ): 12
True Positive (Ασθενείς που βρέθηκαν με Πνευμονία): 378
